In [2]:
from typing import TypedDict
from langgraph.graph import StateGraph,START,END
import re

In [3]:

class TextState(TypedDict):
    rawtext:str
    count:int
    cleaned:str
    summary:str

In [13]:
def clean_text(state:TextState)->dict:
    cleaned=re.sub(r"\s+"," ",state["rawtext"].strip()).lower()
    return {"cleaned":cleaned}


def count_words(state:TextState)->dict:
    return {"count":len(state["cleaned"].split())}

def format_text(state:TextState)->dict:
    preview = state["cleaned"][:50]+("..." if len(state["cleaned"])>50 else "")
    summary=(
        "=== Text Analysis Summary ===\n"
        f"word count : {state["count"]}\n"
        f"preview: {preview}\n"
    )

    return {"summary":summary}

In [15]:
from asyncio import graph
builder=StateGraph(TextState)

builder.add_node("clean_text",clean_text)
builder.add_node("count_words",count_words)
builder.add_node("format_text",format_text)

builder.add_edge(START,"clean_text")
builder.add_edge("clean_text","count_words")
builder.add_edge("count_words","format_text")
builder.add_edge("format_text",END)

graph=builder.compile()

result=graph.invoke({"rawtext":"   The man was in the hurry at his office.    so he got a cap "})

print(result["summary"])


=== Text Analysis Summary ===
word count : 14
preview: the man was in the hurry at his office. so he got ...

